In [2]:
pip install pyodbc pandas sqlalchemy

Note: you may need to restart the kernel to use updated packages.


In [14]:
import os
import urllib.parse
import pandas as pd
import pyodbc
print(pyodbc.drivers())

['SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'ODBC Driver 17 for SQL Server', 'SQL Server Native Client 11.0']


In [ ]:

from sqlalchemy import create_engine


########################################
# تحديث الاتصال مع Azure #
########################################
server = "omar.database.windows.net"  # اكتب اسم السيرفر الخاص بك هنا
database = "testsqlsrver2026"                         # اسم قاعدة البيانات في Azure
username = "omar"                        # اسم المستخدم
password = "*********"                        # كلمة المرور
driver = "{ODBC Driver 17 for SQL Server}"        # برمجية الربط القياسية لـ Azure SQL

# Construct Connection Strings
connection_url = f"mssql+pyodbc://{username}:{password}@{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server"
# Create SQLAlchemy engine for pandas export
engine = create_engine(connection_url)

# Native pyodbc connection for DDL statements (CREATE / DROP)
conn_str = f"DRIVER={driver};SERVER={server};DATABASE={database};UID={username};PWD={password};Encrypt=yes;TrustServerCertificate=no;"
conn = pyodbc.connect(conn_str)
cursor = conn.cursor()
print("Connection established successfully with Azure SQL!")

Connection established successfully with Azure SQL!


In [13]:
# الفانكشن دي عشان تمسح الجدول القديم لو موجود وتكرته من جديد
def drop_recreate(c, tablename, create_query):
    # بنتأكد الأول إذا الجدول موجود في داتابيز Azure عشان نمسحه بدون مشاكل
    c.execute(f"IF OBJECT_ID('dbo.{tablename}', 'U') IS NOT NULL DROP TABLE dbo.{tablename};")
    c.execute(create_query)
    conn.commit()
    print(f"تم إنشاء جدول {tablename} بنجاح.")

# الفانكشن دي بديل copy_from بتاعة بوستجرس، بنقرأ الـ CSV بـ pandas ونرفعه على طول
def populate_table(filename, tablename):
    try:
        df = pd.read_csv(filename)
        # بنرفع الداتا على طول لقاعدة بيانات Azure SQL
        df.to_sql(tablename, con=engine, if_exists='append', index=False)
        print(f"تم رفع بيانات {tablename} بنجاح.")
    except Exception as error:
        print(f"حصلت مشكلة وأنا برفع بيانات {tablename}: {error}")


# 1. تجهيز ورفع جدول الـ Riders
table = "rider"
filename = './riders.csv'
create = """
CREATE TABLE rider (
    rider_id INT PRIMARY KEY, 
    first VARCHAR(50), 
    last VARCHAR(50), 
    address VARCHAR(100), 
    birthday DATE, 
    account_start_date DATE, 
    account_end_date DATE, 
    is_member BIT -- بنستخدم BIT هنا بدل BOOLEAN عشان T-SQL يفهمها
);
"""
drop_recreate(cursor, table, create)
populate_table(filename, table)


# 2. تجهيز ورفع جدول الـ Payments
table = "payment"
filename = './payments.csv'
create = """
CREATE TABLE payment (
    payment_id INT PRIMARY KEY, 
    date DATE, 
    amount DECIMAL(10, 2), -- DECIMAL أضمن وأدق للمبالغ المالية من MONEY
    rider_id INT
);
"""
drop_recreate(cursor, table, create)
populate_table(filename, table)


# 3. تجهيز ورفع جدول الـ Stations
table = "station"
filename = './stations.csv'
create = """
CREATE TABLE station (
    station_id VARCHAR(50) PRIMARY KEY, 
    name VARCHAR(75), 
    latitude FLOAT, 
    longitude FLOAT
);
"""
drop_recreate(cursor, table, create)
populate_table(filename, table)


# 4. تجهيز ورفع جدول الـ Trips
table = "trip"
filename = './trips.csv'
create = """
CREATE TABLE trip (
    trip_id VARCHAR(50) PRIMARY KEY, 
    rideable_type VARCHAR(75), 
    start_at DATETIME2, -- DATETIME2 بتدي دقة أفضل مع التواريخ والأوقات
    ended_at DATETIME2, 
    start_station_id VARCHAR(50), 
    end_station_id VARCHAR(50), 
    rider_id INT
);
"""
drop_recreate(cursor, table, create)
populate_table(filename, table)


# إغلاق الاتصالات بعد ما خلصنا كل شغلنا
cursor.close()
conn.close()

print("\nعاش! كل حاجة اكتملت والبيانات اترفعت بنجاح.")

ProgrammingError: Attempt to use a closed cursor.

In [19]:
#ادخال البيانات # 1. إعداد قائمة الملفات وأسمائها والأعمدة الخاصة بكل ملف
files_config = {
    "rider": {
        "file": "riders.csv",
        "columns": ["rider_id", "first", "last", "address", "birthday", "account_start_date", "account_end_date", "is_member"]
    },
    "payment": {
        "file": "payments.csv",
        "columns": ["payment_id", "date", "amount", "rider_id"]
    },
    "station": {
        "file": "stations.csv",
        "columns": ["station_id", "name", "latitude", "longitude"]
    },
    "trip": {
        "file": "trips.csv",
        "columns": ["trip_id", "rideable_type", "start_at", "ended_at", "start_station_id", "end_station_id", "rider_id"]
    }
}

# 2. اسم المجلد الذي يحتوي على الملفات
folder_name = "data source"

print("بدء إدخال البيانات إلى الجداول...\n")

for table_name, config in files_config.items():
    file_name = config["file"]
    
    # يحدد المسار الرئيسي لكود البايثون يدخل لمجلد data source ثم يجلب الملف
    file_path = os.path.join(os.path.dirname(__file__) if '__file__' in locals() else '.', folder_name, file_name)
    
    if os.path.exists(file_path):
        try:
            print(f"جاري إدخال {file_name} من مجلد {folder_name}...")
            
            # قراءة ملف الـ CSV وتسمية الأعمدة بالترتيب
            df = pd.read_csv(file_path, header=None, names=config["columns"])
            
            # إدخال القيم في الجدول المقابل بالـ engine الخاص بك
            df.to_sql(table_name, con=engine, if_exists='append', index=False, chunksize=10000)
            
            print(f" تم إدخال {len(df)} صف في جدول '{table_name}'.\n")
        except Exception as e:
            print(f" حصلت مشكلة أثناء إدخال '{file_name}': {e}\n")
    else:
        print(f" الملف غير موجود في المسار: {file_path}\n")

print("تم رفع جميع البيانات بنجاح!")

بدء إدخال البيانات إلى الجداول...

جاري إدخال riders.csv من مجلد data source...
 حصلت مشكلة أثناء إدخال 'riders.csv': Can't reconnect until invalid transaction is rolled back.  Please rollback() fully before proceeding (Background on this error at: https://sqlalche.me/e/20/8s2b)

جاري إدخال payments.csv من مجلد data source...
 تم إدخال 1946607 صف في جدول 'payment'.

جاري إدخال stations.csv من مجلد data source...
 تم إدخال 838 صف في جدول 'station'.

جاري إدخال trips.csv من مجلد data source...
 تم إدخال 50000 صف في جدول 'trip'.

تم رفع جميع البيانات بنجاح!


In [20]:
import os
import pandas as pd

# مسار ملف riders.csv
file_path = os.path.join(os.path.dirname(__file__) if '__file__' in locals() else '.', 'data source', 'riders.csv')

if os.path.exists(file_path):
    try:
        print("جاري تجهيز بيانات riders...")
        
        # 1. قراءة الملف بدون هيدر
        cols = ["rider_id", "first", "last", "address", "birthday", "account_start_date", "account_end_date", "is_member"]
        df = pd.read_csv(file_path, header=None, names=cols)
        
        # 2. تحويل القيم الفارغة NaN إلى None ليفهمها SQL كـ NULL
        df = df.object.where(pd.notnull(df), None) if hasattr(df, 'object') else df.where(pd.notnull(df), None)
        
        # 3. تحويل True/False إلى 1/0 لتتناسب مع BIT في Azure SQL
        df['is_member'] = df['is_member'].astype(int)

        # 4. الرفع إلى Azure SQL
        print("جاري رفع البيانات إلى جدول rider...")
        df.to_sql('rider', con=engine, if_exists='append', index=False, chunksize=5000)
        
        print(f" تم رفع {len(df)} صف في جدول 'rider' بنجاح!")

    except Exception as e:
        print(f" حصلت مشكلة أثناء الرفع: {e}")

جاري تجهيز بيانات riders...
جاري رفع البيانات إلى جدول rider...
 تم رفع 75000 صف في جدول 'rider' بنجاح!
